# Create FLUT

The test requires a non-reactive FLUT with variable enthalpy/temperature and constant mixture.

In [ ]:
import pyFLUT
import cantera as ct
import numpy as np

Load mechanism (included in cantera)

In [ ]:
gas = ct.Solution('ch4_smooke_EKT.yaml')

### Create manifold

In [ ]:
Ts = [780,800,820]

# Mass fraction CH4
Z_CH4 = np.linspace(0,0.5,5)

# Nitrogen mass fractions in oxidizer
Y_N2 = 1

Create solution array

In [ ]:
sol_arr = ct.SolutionArray(gas,(len(Z_CH4),len(Ts)))

In [ ]:
species_array = np.zeros((len(Z_CH4),len(Ts),gas.n_species))
species_array[...,gas.species_index('N2')] = (Y_N2 * ( 1 - Z_CH4 ))[:,np.newaxis]
species_array[...,gas.species_index('CH4')] = Z_CH4[:,np.newaxis]

In [ ]:
sol_arr.TPY = Ts, ct.one_atm, species_array

### Build flut from the data

In [ ]:
data = np.concatenate((sol_arr.T[...,np.newaxis],sol_arr.P[...,np.newaxis],sol_arr.density[...,np.newaxis],sol_arr.enthalpy_mass[...,np.newaxis],species_array),axis=-1)

In [ ]:
import collections

input_dict = collections.OrderedDict({'Z':np.array(Z_CH4),'T':np.array(Ts)})
output_list = [(name,i) for i, name in enumerate(['T','p','rho','ha'])]
output_list = output_list + [(name, i + len(output_list)) for i, name in enumerate(gas.species_names)]
output_dict = collections.OrderedDict(output_list)

flut = pyFLUT.Flut(data,input_dict=input_dict,output_dict=output_dict)
flut

In [ ]:
flut.calc_non_dimensional_quantity(along_variable='T',quantity='ha')
flut = flut.map_variables(from_inp='T',to_inp='hanorm')
flut

In [ ]:
flut.output_dict

Calc additionally needed variables and rename (maybe this can be adapted in the future)

In [ ]:
flut.calc_with_cantera(['rho', 'cp', 'lambda', 'visc','alpha'],gas=gas)
flut = flut.rename_variables(rename_dict={'visc':'mu'})

In [ ]:
flut.output_dict

In [ ]:
output_variables = ['T','p','rho','ha','CH4','O2','N2','cp','lambda','mu','alpha','hanorm','ha_min','ha_max','ha']
flut.export_to_openfoam('non_reactive_N2_CH4.h5',cantera_file='ch4_smooke_EKT.yaml',output_variables=output_variables)

### Check 

In [ ]:
read_flut = pyFLUT.Flut.read_h5('non_reactive_N2_CH4.h5')

In [ ]:
read_flut.output_dict